In [575]:
import pandas as pd
import warnings
warnings.simplefilter('ignore')
print('succes')

succes


In [ ]:
import sqlite3
try:
    go_crm_train_conn = sqlite3.connect("go_crm_train.sqlite")
    go_sales_train_conn = sqlite3.connect("go_sales_train.sqlite")
    go_staff_train_conn = sqlite3.connect("go_staff_train.sqlite")
    print('Databases connected succesfully')
except sqlite3.Error as error:
    print('Error occured: ', error)

Databases connected succesfully


In [577]:
import pyodbc

DB  = {
    'servername' : r'LAPTOP-28CE8V5M\SQLEXPRESS',
    'database' : 'Source Data Model'
}

export_conn = pyodbc.connect(
    'DRIVER={SQL SERVER};SERVER=' +
    DB['servername'] + 
    ';DATABASE=' + 
    DB['database'] + 
    ';Trusted_Connection=yes'
)

export_cursor = export_conn.cursor()

print('done')


done


In [578]:
export_cursor.execute("EXEC sp_MSforeachtable 'ALTER TABLE ? NOCHECK CONSTRAINT ALL'")

export_cursor.execute("SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_TYPE = 'BASE TABLE'")
tables = export_cursor.fetchall()

for table in tables:
    table_name = table[0]
    export_cursor.execute(f"DELETE FROM {table_name}")
    print(f"Emptied table: {table_name}")

export_cursor.execute("EXEC sp_MSforeachtable 'ALTER TABLE ? WITH CHECK CHECK CONSTRAINT ALL'")

export_conn.commit()


Emptied table: retailer_contact
Emptied table: age_group
Emptied table: sales_territory
Emptied table: country
Emptied table: retailer_segment
Emptied table: retailer_headquarters
Emptied table: retailer_type
Emptied table: retailer
Emptied table: retailer_site
Emptied table: sales_demographic
Emptied table: sales_branch
Emptied table: sales_staff
Emptied table: order_method
Emptied table: order_header
Emptied table: product_line
Emptied table: product_type
Emptied table: product
Emptied table: order_details
Emptied table: return_reason
Emptied table: returned_item
Emptied table: course
Emptied table: satisfaction_type
Emptied table: satisfaction
Emptied table: training
Emptied table: product_forecast_train
Emptied table: inventory_levels_train


In [579]:
age_group_dimensie = pd.read_sql("SELECT * FROM age_group", go_crm_train_conn)

for index, row in age_group_dimensie.iterrows():
    try:
        query = f"INSERT INTO age_group VALUES ({row['AGE_GROUP_CODE']}, {row['UPPER_AGE']}, {row['LOWER_AGE']})"
        export_cursor.execute(query)
    except pyodbc.Error:
        print(query)


export_conn.commit()

sales_territory_dimensie = pd.read_sql("SELECT * FROM sales_territory", go_crm_train_conn)

for index, row in sales_territory_dimensie.iterrows():
    try:
        query = f"INSERT INTO sales_territory VALUES ({row['SALES_TERRITORY_CODE']}, '{row['TERRITORY_NAME_EN']}')"
        export_cursor.execute(query)
        
    except pyodbc.Error:
        print(query)


export_conn.commit()

country_dimensie1 = pd.read_sql("SELECT * FROM country", go_crm_train_conn)
country_dimensie2 = pd.read_sql("SELECT * FROM country", go_sales_train_conn)

country_dimensies = pd.merge(country_dimensie1, country_dimensie2, 
                     left_on=["COUNTRY_CODE", "COUNTRY_EN"], 
                     right_on=["COUNTRY_CODE", "COUNTRY"], 
                     how="outer")
country_dimensies = country_dimensies.drop(columns=["COUNTRY"])

for index, row in country_dimensies.iterrows():
    try:
        query = f"INSERT INTO country VALUES ({row['COUNTRY_CODE']}, '{row['COUNTRY_EN']}', '{row['FLAG_IMAGE']}', {row['SALES_TERRITORY_CODE']}, '{row['LANGUAGE']}', '{row['CURRENCY_NAME']}')"
        export_cursor.execute(query)
    except pyodbc.Error:
        print(query)


export_conn.commit()

retailer_segment_dimensie = pd.read_sql("SELECT * FROM retailer_segment", go_crm_train_conn)

for index, row in retailer_segment_dimensie.iterrows():
    try:
        query = f"INSERT INTO retailer_segment VALUES ({row['SEGMENT_CODE']}, '{row['LANGUAGE']}', '{row['SEGMENT_NAME']}', '{row['SEGMENT_DESCRIPTION']}')"
        export_cursor.execute(query)
        
    except pyodbc.Error:
        print(query)


export_conn.commit()

retailer_headquarters_dimensie = pd.read_sql("SELECT * FROM retailer_headquarters", go_crm_train_conn)

for index, row in retailer_headquarters_dimensie.iterrows():
    try:
        query = f"INSERT INTO retailer_headquarters VALUES ({row['RETAILER_CODEMR']}, '{row['RETAILER_NAME'].replace("'", "''")}', '{row['ADDRESS1'].replace("'", "''")}', '{row['ADDRESS2'].replace("'", "''") if (row['ADDRESS2'] != None) else row['ADDRESS2']}', '{row['CITY']}', '{row['REGION']}', {row['COUNTRY_CODE']}, '{row['PHONE']}', '{row['FAX']}', {row['SEGMENT_CODE']} )"
        export_cursor.execute(query)
        
    except pyodbc.Error:
        print(query)


export_conn.commit()

retailer_type_dimensie = pd.read_sql("SELECT * FROM retailer_type", go_crm_train_conn)

for index, row in retailer_type_dimensie.iterrows():
    try:
        query = f"INSERT INTO retailer_type VALUES ({row['RETAILER_TYPE_CODE']}, '{row['RETAILER_TYPE_EN']}')"
        export_cursor.execute(query)
        
    except pyodbc.Error:
        print(query)


export_conn.commit()

retailer_dimensie = pd.read_sql("SELECT * FROM retailer", go_crm_train_conn)

for index, row in retailer_dimensie.iterrows():
    try:
        query = f"INSERT INTO retailer VALUES ({row['RETAILER_CODE']}, {row['RETAILER_CODEMR'] if (row['RETAILER_CODEMR'] > 0) else 'null'}, '{row['COMPANY_NAME'].replace("'", "''")}', {row['RETAILER_TYPE_CODE']})"
        export_cursor.execute(query)
        
    except pyodbc.Error:
        print(query)


export_conn.commit()

retailer_site_dimensie = pd.read_sql("SELECT * FROM retailer_site", go_crm_train_conn)

for index, row in retailer_site_dimensie.iterrows():
    try:
        query = f"INSERT INTO retailer_site VALUES ({row['RETAILER_SITE_CODE']}, {row['RETAILER_CODE']}, '{row['ADDRESS1'].replace("'", "''")}', '{row['ADDRESS2'].replace("'", "''") if (row['ADDRESS2'] != None) else row['ADDRESS2']}', '{row['CITY'].replace("'", "''")}', '{row['REGION']}', {row['COUNTRY_CODE']}, {row['ACTIVE_INDICATOR']} )"
        export_cursor.execute(query)
        
    except pyodbc.Error:
        print(query)


export_conn.commit()

retailer_contact_dimensie = pd.read_sql("SELECT * FROM retailer_contact", go_crm_train_conn)

for index, row in retailer_contact_dimensie.iterrows():
    try:
        query = f"INSERT INTO retailer_contact VALUES ({row['RETAILER_CONTACT_CODE']}, {row['RETAILER_SITE_CODE']}, '{row['FIRST_NAME'].replace("'", "''")}', '{row['LAST_NAME'].replace("'", "''")}', '{row['JOB_POSITION_EN']}', {row['EXTENSION'] if (row['EXTENSION'] > 0) else 'null'}, '{row['FAX']}', '{row['E_MAIL']}', '{row['GENDER']}')"
        export_cursor.execute(query)
        
    except pyodbc.Error:
        print(query)


export_conn.commit()

sales_demographic_dimensie = pd.read_sql("SELECT * FROM sales_demographic", go_crm_train_conn)

for index, row in sales_demographic_dimensie.iterrows():
    try:
        query = f"INSERT INTO sales_demographic VALUES ({row['DEMOGRAPHIC_CODE']}, {row['RETAILER_CODEMR']}, {row['AGE_GROUP_CODE']}, {row['SALES_PERCENT']})"
        export_cursor.execute(query)
        
    except pyodbc.Error:
        print(query)


export_conn.commit()

sales_branch_dimensie = pd.read_sql("SELECT * FROM sales_branch", go_sales_train_conn)

for index, row in sales_branch_dimensie.iterrows():
    try:
        query = f"INSERT INTO sales_branch VALUES ({row['SALES_BRANCH_CODE']}, '{row['ADDRESS1'].replace("'", "''")}', '{row['ADDRESS2'].replace("'", "''") if (row['ADDRESS2'] != None) else row['ADDRESS2']}', '{row['CITY'].replace("'", "''")}', '{row['REGION']}', {row['COUNTRY_CODE']} )"
        export_cursor.execute(query)
        
    except pyodbc.Error:
        print(query)


export_conn.commit()

sales_staff_dimensie1 = pd.read_sql("SELECT * FROM sales_staff", go_sales_train_conn)
sales_staff_dimensie2 = pd.read_sql("SELECT * FROM sales_staff", go_staff_train_conn)

sales_staff_dimensies = pd.merge(sales_staff_dimensie1, sales_staff_dimensie2, 
                     left_on=["SALES_STAFF_CODE", "FIRST_NAME", "LAST_NAME", "POSITION_EN", "WORK_PHONE", "EXTENSION", "FAX", "EMAIL", "DATE_HIRED", "SALES_BRANCH_CODE"], 
                     right_on=["SALES_STAFF_CODE", "FIRST_NAME", "LAST_NAME", "POSITION_EN", "WORK_PHONE", "EXTENSION", "FAX", "EMAIL", "DATE_HIRED", "SALES_BRANCH_CODE"], 
                     how="outer")

for index, row in sales_staff_dimensies.iterrows():
    try:
        query = f"INSERT INTO sales_staff VALUES ({row['SALES_STAFF_CODE']}, '{row['FIRST_NAME'].replace("'", "''")}', '{row['LAST_NAME'].replace("'", "''")}', '{row['POSITION_EN'].replace("'", "''")}', '{row['WORK_PHONE']}', {row['EXTENSION'] if (row['EXTENSION'] > 0) else 'null'}, '{row['FAX']}', '{row['EMAIL']}', '{row['DATE_HIRED']}', {row['SALES_BRANCH_CODE']} , {row['MANAGER_CODE'] if (row['MANAGER_CODE'] != None) else 'null'})"
        export_cursor.execute(query)
        
    except pyodbc.Error:
        print(query)


export_conn.commit()

order_method_dimensie = pd.read_sql("SELECT * FROM order_method", go_sales_train_conn)

for index, row in order_method_dimensie.iterrows():
    try:
        query = f"INSERT INTO order_method VALUES ({row['ORDER_METHOD_CODE']}, '{row['ORDER_METHOD_EN']}')"
        export_cursor.execute(query)
        
    except pyodbc.Error:
        print(query)


export_conn.commit()

order_header_dimensie = pd.read_sql("SELECT * FROM order_header", go_sales_train_conn)

for index, row in order_header_dimensie.iterrows():
    try:
        query = f"INSERT INTO order_header VALUES ({row['ORDER_NUMBER']}, '{row['RETAILER_NAME'].replace("'", "''")}', {row['RETAILER_SITE_CODE']}, {row['RETAILER_CONTACT_CODE'] if(row['RETAILER_CONTACT_CODE'] > 0) else 'null'}, {row['SALES_STAFF_CODE']}, {row['SALES_BRANCH_CODE']}, '{row['ORDER_DATE']}', {row['ORDER_METHOD_CODE']})"
        export_cursor.execute(query)
        
    except pyodbc.Error:
        print(query)


export_conn.commit()

product_line_dimensie = pd.read_sql("SELECT * FROM product_line", go_sales_train_conn)

for index, row in product_line_dimensie.iterrows():
    try:
        query = f"INSERT INTO product_line VALUES ({row['PRODUCT_LINE_CODE']}, '{row['PRODUCT_LINE_EN']}')"
        export_cursor.execute(query)
        
    except pyodbc.Error:
        print(query)


export_conn.commit()

product_type_dimensie = pd.read_sql("SELECT * FROM product_type", go_sales_train_conn)

for index, row in product_type_dimensie.iterrows():
    try:
        query = f"INSERT INTO product_type VALUES ({row['PRODUCT_TYPE_CODE']}, {row['PRODUCT_LINE_CODE']}, '{row['PRODUCT_TYPE_EN']}')"
        export_cursor.execute(query)
        
    except pyodbc.Error:
        print(query)


export_conn.commit()

product_dimensie = pd.read_sql("SELECT * FROM product", go_sales_train_conn)

for index, row in product_dimensie.iterrows():
    try:
        query = f"INSERT INTO product VALUES ({row['PRODUCT_NUMBER']}, '{row['INTRODUCTION_DATE']}', {row['PRODUCT_TYPE_CODE']}, {row['PRODUCTION_COST']}, {row['MARGIN']}, '{row['PRODUCT_IMAGE']}', '{row['LANGUAGE']}', '{row['PRODUCT_NAME'].replace("'", "''")}', '{row['DESCRIPTION'].replace("'", "''")}')"
        export_cursor.execute(query)
        
    except pyodbc.Error:
        print(query)


export_conn.commit()

order_details_dimensie = pd.read_sql("SELECT * FROM order_details", go_sales_train_conn)

for index, row in order_details_dimensie.iterrows():
    try:
        query = f"INSERT INTO order_details VALUES ({row['ORDER_DETAIL_CODE']}, {row['ORDER_NUMBER']}, {row['PRODUCT_NUMBER']}, {row['QUANTITY']}, {row['UNIT_COST']}, {row['UNIT_PRICE']}, {row['UNIT_SALE_PRICE']})"
        export_cursor.execute(query)
        
    except pyodbc.Error:
        print(query)


export_conn.commit()

return_reason_dimensie = pd.read_sql("SELECT * FROM return_reason", go_sales_train_conn)

for index, row in return_reason_dimensie.iterrows():
    try:
        query = f"INSERT INTO return_reason VALUES ({row['RETURN_REASON_CODE']}, '{row['RETURN_DESCRIPTION_EN']}')"
        export_cursor.execute(query)
        
    except pyodbc.Error:
        print(query)


export_conn.commit()

returned_item_dimensie = pd.read_sql("SELECT * FROM returned_item", go_sales_train_conn)

for index, row in returned_item_dimensie.iterrows():
    try:
        query = f"INSERT INTO returned_item VALUES ({row['RETURN_CODE']}, '{row['RETURN_DATE']}', {row['ORDER_DETAIL_CODE']}, {row['RETURN_REASON_CODE']}, {row['RETURN_QUANTITY']})"
        export_cursor.execute(query)
        
    except pyodbc.Error:
        print(query)


export_conn.commit()

course_dimensie = pd.read_sql("SELECT * FROM course", go_staff_train_conn)

for index, row in course_dimensie.iterrows():
    try:
        query = f"INSERT INTO course VALUES ({row['COURSE_CODE']}, '{row['COURSE_DESCRIPTION']}')"
        export_cursor.execute(query)
        
    except pyodbc.Error:
        print(query)


export_conn.commit()

satisfaction_type_dimensie = pd.read_sql("SELECT * FROM satisfaction_type", go_staff_train_conn)

for index, row in satisfaction_type_dimensie.iterrows():
    try:
        query = f"INSERT INTO satisfaction_type VALUES ({row['SATISFACTION_TYPE_CODE']}, '{row['SATISFACTION_TYPE_DESCRIPTION']}')"
        export_cursor.execute(query)
        
    except pyodbc.Error:
        print(query)


export_conn.commit()

satisfaction_dimensie = pd.read_sql("SELECT * FROM satisfaction", go_staff_train_conn)

for index, row in satisfaction_dimensie.iterrows():
    try:
        query = f"INSERT INTO satisfaction VALUES ({row['YEAR']}, {row['SALES_STAFF_CODE']}, {row['SATISFACTION_TYPE_CODE']})"
        export_cursor.execute(query)
        
    except pyodbc.Error:
        print(query)


export_conn.commit()

training_dimensie = pd.read_sql("SELECT * FROM training", go_staff_train_conn)

for index, row in training_dimensie.iterrows():
    try:
        query = f"INSERT INTO training VALUES ({row['YEAR']}, {row['SALES_STAFF_CODE']}, {row['COURSE_CODE']})"
        export_cursor.execute(query)
        
    except pyodbc.Error:
        print(query)


export_conn.commit()

inventory_levels_train_dimensie = pd.read_csv("inventory_levels_train.csv")

for index, row in inventory_levels_train_dimensie.iterrows():
    try:
        query = f"INSERT INTO inventory_levels_train VALUES ({row['PRODUCT_NUMBER']}, {row['INVENTORY_YEAR']}, {row['INVENTORY_MONTH']}, {row['INVENTORY_COUNT']})"
        export_cursor.execute(query)
        
    except pyodbc.Error:
        print(query)


export_conn.commit()

product_forecast_train_dimensie = pd.read_csv("product_forecast_train.csv")

for index, row in product_forecast_train_dimensie.iterrows():
    try:
        query = f"INSERT INTO product_forecast_train VALUES ({row['PRODUCT_NUMBER']}, {row['YEAR']}, {row['MONTH']}, {row['EXPECTED_VOLUME']})"
        export_cursor.execute(query)
        
    except pyodbc.Error:
        print(query)


export_conn.commit()
export_conn.close()